# Standard track: simulating quantum dynamics you can't check

Simulating how a quantum system evolves in time is one of the most promising uses of quantum computers, because classical computers choke on it as systems grow. But that creates an awkward problem: **once a quantum simulation is beyond classical reach, how do you know its answer is right?**

This is not hypothetical. In 2023 IBM reported 127-qubit dynamics "beyond brute-force classical computation"; within weeks, several groups reproduced the results with classical methods. D-Wave (2025) and Quantinuum (2026) reported spin-dynamics simulations at the edge of classical reach, and classical challengers followed again. In 2026 an IBM team published a framework built entirely around this question: *how to estimate an observable, with error bars you can defend, when no classical answer is available* (references at the end).

You will face the same question on IonQ's **Forte 1** trapped-ion computer, in three stages:

| Stage | System | Qubits | Maps onto the hardware... | Exact answer known? |
|---|---|---|---|---|
| 1 | Transverse-field Ising chain | 12 | natively: one `rxx` per bond | **yes**: compute it yourself |
| 2 | XXZ Heisenberg chain | 12 | badly: three entangling gates per bond | **yes** |
| 3 | 2D transverse-field Ising, 6 × 6 lattice | 36 | natively | **no**: that is the point |

In every stage the task is the same: **estimate the observable at each time on the grid as accurately as you can, with an honest error bar.** Stages 1 and 2 let you learn what your methods actually achieve on real hardware, because you can check them. Stage 3 asks you to deliver an answer *and the argument for trusting it*, without a reference to compare to.

### Rules

- **Machine:** IonQ Forte 1 (`qpu.forte-1`). Free practice on IonQ's cloud simulator (ideal or `forte-1` noise model, at most 29 qubits) and on your own laptop.
- **Gates:** anything IonQ accepts: the QIS gates (`rx`, `ry`, `rz`, `rxx`, `rzz`, ...) or IonQ's native gates (`GPI`, `GPI2`, `ZZ`). The circuits you run must be built by your team.
- **No debiasing.** IonQ's built-in error mitigation is not allowed; `run_without_debiasing` switches it off. **Your own** error mitigation (zero-noise extrapolation, symmetry checks, readout correction, ...) is encouraged, as long as you can explain it and report its cost.
- **Budget:** your team project `std-<your team ID>` has a hard dollar cap (about **$11,000**; IonQ refuses jobs beyond it). Half is available from the start, the second half is released at midnight, so nobody can flood the queue early. Section 1 prints what a job costs.
- **Time is the real constraint.** IonQ's queue can take hours, and all teams share the machine. Plan, rehearse on simulators, then submit in batches early.
- **Hand in** one results file per stage (Section 6) and present your work (about 10 minutes plus questions; the organizers confirm the format at the event).

## 0. Setup

1. **Python environment** (from the repository root):

   ```bash
   python -m venv .venv && source .venv/bin/activate     # Windows: .venv\Scripts\activate
   python -m pip install -r requirements.txt
   ```
2. **IonQ account:** accept the invitation email from IonQ and sign in at [cloud.ionq.com](https://cloud.ionq.com) with the invited address. Check that you see your team's project `std-<your team ID>` at [cloud.ionq.com/projects](https://cloud.ionq.com/projects).
3. **API key:** at [cloud.ionq.com/settings/keys](https://cloud.ionq.com/settings/keys), create a key **in your team's project** (not your personal workspace, which has no budget). IonQ shows it only once.
4. **Store it** in a file named `.env` in the repository root: one line, `IONQ_API_KEY=...`. `.env` is git-ignored. Never paste the key into a notebook cell or commit it: anyone with the key can spend your budget.

Everything in this notebook up to Section 1 runs without a key.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
from qiskit_aer import AerSimulator

from dynamics_challenge import (
    PROBLEMS,
    compare_with_exact,
    exact_series,
    forte_like_noise_model,
    hamiltonian,
    noiseless_value,
    series_circuits,
    series_from_counts,
    steps_for_time,
    trotter_circuit,
    write_results,
)
from dynamics_challenge.ionq import (
    IONQ_NOISE_MODEL,
    IONQ_QPU,
    IONQ_SIMULATOR,
    estimate_cost,
    gate_counts,
    load_job_log,
    record_job,
    run_without_debiasing,
)

warnings.filterwarnings('ignore', module='qiskit_ionq')
for key, problem in PROBLEMS.items():
    print(f'{key:<11} {problem.num_qubits:>2} qubits  {problem.hamiltonian_text()}')
    print(f'            start |{problem.initial_bits}> (qubit 0 first), observable: {problem.observable}, '
          f'times {problem.times[0]}..{problem.times[-1]}')

### The three problems

All couplings are fixed in `dynamics_challenge/problems.py`; please don't change them.

- **Stage 1**: $H = J\sum_{i} X_iX_{i+1} + h\sum_i Z_i$ on an open chain of 12 qubits, $J=h=1$, starting from $|000\dots0\rangle$. Observable: the magnetization $m_z(t)=\frac1N\sum_i\langle Z_i\rangle$ at $t = 0.25, 0.5, \dots, 3.0$.
- **Stage 2**: $H = J\sum_i (X_iX_{i+1} + Y_iY_{i+1} + \Delta\, Z_iZ_{i+1})$ on 12 qubits, $J=1$, $\Delta=0.5$, starting from the Néel state $|0101\dots\rangle$. Observable: the staggered magnetization $\frac1N\sum_i(-1)^i\langle Z_i\rangle$, same times.
- **Stage 3**: the stage-1 Hamiltonian on a 6 × 6 open square lattice (36 qubits, 60 bonds), from $|00\dots0\rangle$, observable $m_z(t)$ at $t=0.25,\dots,1.5$. `stage3_3x3` and `stage3_4x4` are the same model on lattices small enough to solve exactly: use them to test your stage-3 methods.

Why these? $e^{-i\theta X_iX_j}$ is exactly IonQ's entangling interaction (`rxx(2θ)`), and $e^{-i\theta Z_i}$ is a free `rz`. So the Ising model is *native*. The Heisenberg bond $XX+YY+\Delta ZZ$ needs an entangling gate for each of the three terms, and the three terms don't commute with their neighbours: *non-native*, and much harder.

The exact answers for the checkable problems take about a second each:

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, key in zip(axes, ['stage1', 'stage2', 'stage3_4x4']):
    p = PROBLEMS[key]
    fine = np.linspace(0, p.times[-1], 121)
    ax.plot(fine, exact_series(p, fine), color='0.3', label='exact')
    ax.plot(p.times, exact_series(p), 'o', color='C0', label='time grid')
    ax.set(title=p.title.split(':')[0] + f': {key}', xlabel='time t', ylabel=p.observable)
    ax.legend()
plt.tight_layout()
plt.show()

### A baseline: product formulas (Trotterization)

Hardware can't apply $e^{-iHt}$ directly. The standard approach splits it into short steps, $e^{-iHt}\approx\big(\prod_k e^{-iH_k\,\delta t}\big)^{t/\delta t}$, and applies each piece as gates. `trotter_circuit(problem, t, steps)` builds this plainly, in `rx`/`ry`/`rz`/`rxx`. More steps mean a smaller **Trotter error** but a deeper circuit, so more **hardware error**. Finding the best compromise, and beating it, is a large part of the challenge.

The cell below shows the trade-off without any noise: the error from Trotterization alone, and how many entangling gates it costs.

In [ ]:
p = PROBLEMS['stage1']
t = 3.0
exact_value = exact_series(p, [t])[0]
print(f'Stage 1 at t = {t}: exact m_z = {exact_value:.4f}')
for steps in (3, 6, 12, 24):
    circuit = trotter_circuit(p, t, steps)
    oneq, twoq = gate_counts(circuit)
    print(f'  {steps:>2} steps: Trotter-only error {noiseless_value(p, circuit) - exact_value:+.4f}   '
          f'{twoq:>3} two-qubit gates, {oneq:>3} one-qubit gates')

### Rehearsing with noise, for free

Forte 1's median two-qubit gate fidelity is about 99.6%. After 130 two-qubit gates, roughly $0.996^{130}\approx 60\%$ of runs are error-free. `forte_like_noise_model()` is a deliberately crude local model built from the published median fidelities (depolarizing gate errors plus readout errors). Real hardware also has coherent errors, crosstalk and drift, so treat it as a rehearsal tool, not a prediction. IonQ's cloud simulator with `noise_model='forte-1'` is a second free opinion (Section 1).

Here is the plain baseline (step size 0.25) for stage 1 on the local model:

In [ ]:
p = PROBLEMS['stage1']
local = AerSimulator(noise_model=forte_like_noise_model(), seed_simulator=7)
result = local.run(series_circuits(p, max_dt=0.25), shots=500).result()
values, errors = series_from_counts(p, [result.get_counts(i) for i in range(len(p.times))])

plt.figure(figsize=(7, 3.6))
fine = np.linspace(0, p.times[-1], 121)
plt.plot(fine, exact_series(p, fine), color='0.3', label='exact')
plt.errorbar(p.times, values, yerr=errors, fmt='o', color='C3', label='baseline, local noise model (500 shots)')
plt.xlabel('time t'); plt.ylabel('m_z'); plt.legend(); plt.title('Stage 1 rehearsal')
plt.show()
print('Note the error bars: they only cover shot noise. The bias is much larger.')

## 1. Connect to IonQ, check costs, and run

The next cell reads your key from `.env`. If none is found, the IonQ cells below are skipped, so you can still work through the rest of the notebook.

**How IonQ bills:** each circuit costs `max(minimum, shots × (1q gates × rate₁ + 2q gates × rate₂))`, and a job of several circuits costs the sum. IonQ bills the circuit after its own compilation, so treat estimates as guides. The cost-check cell prints your organization's actual rates.

In [ ]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
provider = simulator = qpu = None
if os.getenv('IONQ_API_KEY'):
    from qiskit_ionq import IonQProvider
    provider = IonQProvider(os.environ['IONQ_API_KEY'])
    simulator = provider.get_backend(IONQ_SIMULATOR)
    qpu = provider.get_backend(IONQ_QPU)
    print('Connected. QPU:', IONQ_QPU)
else:
    print('No IONQ_API_KEY found (see Section 0). IonQ cells will be skipped.')

In [ ]:
# Free: what would the stage-1 baseline series cost? Nothing is submitted.
SHOTS = 500
if qpu is None:
    print('Skipped (no API key).')
else:
    circuits = series_circuits(PROBLEMS['stage1'], max_dt=0.25)
    estimates = [estimate_cost(qpu, c, SHOTS) for c in circuits]
    print('Rate card:', estimates[0].get('rate_card'))
    total = sum(float(e.get('estimated_total_cost') or 0) for e in estimates)
    print(f"Stage-1 baseline series, {len(circuits)} circuits x {SHOTS} shots: about {total:,.2f} "
          f"{estimates[0].get('estimated_unit', '')}")
    calibration = qpu.calibration()
    if calibration is not None:
        print(f'Forte 1 calibration of {calibration.date:%Y-%m-%d}: {calibration.fidelity}')

In [ ]:
# Free: the same series on IonQ's cloud simulator with the forte-1 noise model (at most 29 qubits).
RUN_CLOUD_SIM = False
if simulator is None or not RUN_CLOUD_SIM:
    print('Skipped. Set RUN_CLOUD_SIM = True (needs an API key).')
else:
    p = PROBLEMS['stage1']
    job = simulator.run(series_circuits(p), shots=500, noise_model=IONQ_NOISE_MODEL)
    record_job(job, simulator.name, 'stage1 baseline (forte-1 noise model)', 500, noise_model=IONQ_NOISE_MODEL)
    result = job.result()
    cloud_values, cloud_errors = series_from_counts(p, [result.get_counts(i) for i in range(len(p.times))])
    print(np.round(cloud_values, 3))

In [ ]:
# Costs money: submit one series to Forte 1, debiasing off. The job ID goes to ionq_jobs.json.
SUBMIT_TO_QPU = False
STAGE, SHOTS, MAX_DT = 'stage1', 500, 0.25
if qpu is None or not SUBMIT_TO_QPU:
    print('Nothing submitted. Set SUBMIT_TO_QPU = True on purpose (it spends your budget).')
else:
    p = PROBLEMS[STAGE]
    job = run_without_debiasing(qpu, series_circuits(p, max_dt=MAX_DT), shots=SHOTS)
    entry = record_job(job, qpu.name, f'{STAGE} baseline series', SHOTS, stage=STAGE, max_dt=MAX_DT,
                       times=list(p.times))
    print('Submitted', entry['job_id'], '- collect it with the next cell, possibly hours from now.')

In [ ]:
# Rerun any time: status of every logged job, and the series of each finished one.
if provider is None:
    print('Skipped (no API key).')
for entry in (load_job_log() if provider else []):
    job = provider.get_backend(entry['backend']).retrieve_job(entry['job_id'])
    status = job.status().name
    print(f"{entry['submitted_at']}  {entry['label']:<40} {status:<10} {entry['job_id']}")
    if status == 'DONE' and 'stage' in entry:
        p = PROBLEMS[entry['stage']]
        result = job.result()
        values, errors = series_from_counts(p, [result.get_counts(i) for i in range(len(entry['times']))])
        print('   values:', np.round(values, 3))

### Running circuits, step by step

Every experiment in this challenge follows the same five steps. The cells below walk through them once on the 3 × 3 rehearsal lattice, so you can copy the pattern. They use Forte 1 if you set `USE_QPU = True`, IonQ's cloud simulator if you have a key, and the local noise model otherwise.

1. **Build** the circuits, one per time point.
2. **Inspect what will actually run.** IonQ recompiles your circuit, so look at the gate counts (they set the cost and the noise) before you pay for anything.
3. **Run** them as one job.
4. **Save the raw counts** to disk immediately, with enough metadata to know later what they were.
5. **Analyze** from the saved file, never from a variable that disappears when the kernel restarts.

In [ ]:
from dynamics_challenge.ionq import load_counts, save_counts

USE_QPU = False   # True spends money: one job of 6 circuits on Forte 1

if USE_QPU and qpu is not None:
    backend, run = qpu, lambda circuits, shots: run_without_debiasing(qpu, circuits, shots)
elif simulator is not None:
    backend, run = simulator, lambda circuits, shots: simulator.run(circuits, shots=shots, noise_model=IONQ_NOISE_MODEL)
else:
    backend, run = local, lambda circuits, shots: local.run(circuits, shots=shots)
print('Running on:', backend.name)

In [ ]:
# 1. Build: one baseline circuit per time point on the 3 x 3 rehearsal lattice.
p = PROBLEMS['stage3_3x3']
circuits = series_circuits(p, max_dt=0.25)

# 2. Inspect: what will this cost in gates? (IonQ's compiler may change the counts.)
for t, c in zip(p.times, circuits):
    oneq, twoq = gate_counts(c)
    print(f't = {t:<5} {twoq:>3} two-qubit gates, {oneq:>3} one-qubit gates')
if backend is qpu:
    estimate = sum(float(estimate_cost(qpu, c, 500).get('estimated_total_cost') or 0) for c in circuits)
    print(f'Estimated cost at 500 shots: ${estimate:,.2f}')
circuits[0].draw('mpl', fold=-1)   # the first time point: one Trotter step

In [ ]:
# 3. Run all time points as one job, then 4. save the raw counts straight away.
SHOTS = 500
job = run(circuits, SHOTS)
if backend is not local:
    record_job(job, backend.name, 'how-to: stage3_3x3 baseline', SHOTS, stage=p.key, times=list(p.times))
result = job.result()   # waits; for a QPU job this can take hours (collect it later with Section 1's cell)
counts_list = [result.get_counts(i) for i in range(len(circuits))]
save_counts(f'data/{p.key}_baseline_{backend.name}.json', counts_list,
            stage=p.key, times=list(p.times), shots=SHOTS, backend=backend.name, max_dt=0.25,
            job_id=job.job_id() if backend is not local else None)
print('Saved', len(counts_list), 'counts dictionaries.')

In [ ]:
# 5. Analyze from the saved file.
saved = load_counts(f'data/{p.key}_baseline_{backend.name}.json')
values, errors = series_from_counts(p, saved['counts'])
fine = np.linspace(0, p.times[-1], 121)
plt.figure(figsize=(7, 3.6))
plt.plot(fine, exact_series(p, fine), color='0.3', label='exact')
plt.errorbar(saved['times'], values, yerr=errors, fmt='o', color='C3', label=f"baseline on {saved['backend']}")
plt.xlabel('time t'); plt.ylabel('m_z'); plt.legend(); plt.title('3 x 3 rehearsal')
plt.show()

## 2. Stage 1: get the native problem right

You can check every answer here, so use stage 1 to find out **what your methods really achieve on Forte 1**, including how wrong your error bars are. Things worth investigating:

1. **Trotter error vs. gate error.** For each time, which step size minimizes the *total* error? Does a higher-order formula help? (Try `order=2` on stage 1 and explain the surprising result.)
2. **Shorter circuits for the same physics.** The exact answer is a function of $J\,t$ and $h\,t$ only; what freedom does that leave you? Can two layers of single-qubit gates be merged? Do you need all the gates near the end of the circuit, given that you only measure $Z$?
3. **Error mitigation you design yourself:**
   - **Readout correction.** Measure how often a prepared 0 reads as 1 and vice versa.
   - **Zero-noise extrapolation.** Amplify the noise on purpose, for example with `rxx(θ) rxx(-θ) rxx(θ)` folding, then extrapolate back to zero noise. What does each point cost you?
   - **Rescaling** by a decay factor measured on a circuit whose answer you know.
4. **Uncertainty.** Shot noise is the *smallest* part of your error. How do you put an honest error bar on a mitigated estimate?

Use `compare_with_exact(results)` to check a results file: it reports the RMS error and the fraction of points whose exact value falls within two of your stated error bars. If your error bars are honest, that fraction is about 95%.

## 3. Stage 2: the same size, a much harder mapping

Each Heisenberg bond needs an `XX`, a `YY` *and* a `ZZ` rotation, so the baseline circuit has three times as many entangling gates as stage 1 for the same time step, and it also needs smaller steps. Rehearse the baseline on the local noise model and watch it fail:

In [ ]:
p = PROBLEMS['stage2']
result = local.run(series_circuits(p, max_dt=0.25), shots=500).result()
values, errors = series_from_counts(p, [result.get_counts(i) for i in range(len(p.times))])
fine = np.linspace(0, p.times[-1], 121)
plt.figure(figsize=(7, 3.6))
plt.plot(fine, exact_series(p, fine), color='0.3', label='exact')
plt.errorbar(p.times, values, yerr=errors, fmt='o', color='C3', label='baseline, local noise model')
plt.xlabel('time t'); plt.ylabel('staggered magnetization'); plt.legend(); plt.title('Stage 2 rehearsal')
plt.show()
oneq, twoq = gate_counts(series_circuits(p)[-1])
print(f'Deepest baseline circuit: {twoq} two-qubit gates')

Questions worth your time:

- **How cheap can one bond be?** What is the minimum number of entangling gates for $e^{-i\theta(XX+YY+\Delta ZZ)}$? Look up the "canonical" (KAK / Cartan) decomposition of two-qubit gates. Can you do better than three separate rotations?
- **Order of the terms.** Which orderings of the bond terms within a step cancel errors, and which make them worse?
- **Use the physics.** This Hamiltonian conserves the total magnetization $\sum_i Z_i$. Your Néel start has total magnetization 0, and so must every *ideal* outcome. What does that let you do with each measured bitstring? What does it cost (in shots), and what does it miss?
- **Compare with stage 1.** At equal error, how much deeper can you afford to go? Present that comparison: it shows how much the hardware's native interaction shapes what is practical.

## 4. Stage 3: the problem nobody can check for you

Stage 3 is 36 qubits: beyond a laptop's exact simulation (the state vector has $2^{36}\approx 7\times10^{10}$ amplitudes) and beyond IonQ's 29-qubit cloud simulator. You still have to deliver $m_z(t)$ with error bars, and **convince a skeptic that they are right**.

There is no single known recipe. These are the directions the research literature takes; working out which apply here and how, is your job:

1. **Rehearse where you can check.** `stage3_3x3` (9 qubits) and `stage3_4x4` (16 qubits) are the same model. Run your exact pipeline on them, on hardware too, and see how its error changes with size. Is 6 × 6 more like 4 × 4 or different? Why?
2. **Locality and light cones.** After a few Trotter steps, a single-site $\langle Z_i\rangle$ depends only on qubits within a few lattice spacings of $i$. Does that make some stage-3 quantities classically computable after all? For which times?
3. **Circuits with known answers that look like yours.** At special angles every gate becomes a *Clifford* gate, and Clifford circuits of any size are easy to simulate classically. Can you build circuits with the same structure as your stage-3 circuits but a known answer, and use them to measure how much the hardware damps your signal?
4. **Echoes.** Running $U$ and then $U^\dagger$ should return you to the start. How far you miss tells you about the noise in $U$. How would you turn that into a correction or an error bar, and what can it *not* detect?
5. **Symmetries.** The Ising Hamiltonian commutes with the global parity $\prod_i Z_i$. What does that let you check?
6. **Internal consistency.** Do different step sizes, different decompositions or different mitigation settings agree within their error bars? Agreement is evidence. Disagreement is information.
7. **Classical approximations.** Tensor networks and Pauli propagation are what the rebuttal papers used. Even a rough classical estimate at early times is a useful cross-check, if you understand its own errors.

Whatever you do, the final answer is a results file **plus a verification argument**: *why should we believe these numbers, and how far?* The organizers may compare your stage-3 numbers with a high-accuracy classical computation after the event, so an honest error bar is worth more than a confident wrong one.

## 5. Spending the budget wisely

Rough costs of the plain baselines (step size 0.25, debiasing off) at our organization's Forte rates. Section 1 prints the current rates.

| One complete time series | circuits | about |
|---|---|---|
| Stage 1, 500 shots per time | 12 | $460 |
| Stage 2, 500 shots per time | 12 | $1,570 |
| Stage 3, 300 shots per time | 6 | $360 |
| Stage 3, 1000 shots per time | 6 | $1,200 |

Your own circuits should be cheaper than these baselines, and mitigation multiplies the number of circuits. Two facts change how you plan:

- **Every circuit costs at least a minimum** (about $20). Many tiny circuits are expensive; batch your circuits into one job, and use enough shots to make each circuit count.
- **The queue is the scarcer resource.** Submit a full batch, then work on the next idea while it runs. Leave time for one more round after you see the hardware data.

## 6. Hand in your results

One file per stage, made with `write_results` (which checks the format). For each time on the grid: your value, your honest one-standard-deviation uncertainty, and the IonQ job IDs it came from, plus a few sentences on your method. The example below writes a file from the local rehearsal so you can see the format. **Replace it with your hardware results.**

In [ ]:
p = PROBLEMS['stage1']
result = local.run(series_circuits(p), shots=500).result()
values, errors = series_from_counts(p, [result.get_counts(i) for i in range(len(p.times))])
example = write_results(
    'stage1_results_EXAMPLE.json',
    team_id='your_team_id',
    stage='stage1',
    points=[{'time': t, 'value': float(v), 'uncertainty': float(e), 'job_ids': ['local-rehearsal']}
            for t, v, e in zip(p.times, values, errors)],
    method='EXAMPLE ONLY: plain first-order Trotter, dt = 0.25, local noise model, shot-noise error bars.',
)
report = compare_with_exact(example)
print(f"RMS error {report['rms_error']:.3f}; exact value within 2 sigma at "
      f"{report['fraction_within_2_sigma']:.0%} of the times (honest error bars give about 95%)")

### What the judges look for

- **Accuracy, honestly reported** (stages 1 and 2): how close you got, *and* whether your error bars covered the truth. A team whose error bars are honest beats a team with slightly smaller errors and dishonest error bars.
- **The verification argument** (stage 3): how convincingly you justify your numbers without a reference. Concrete checks with data beat general claims.
- **Understanding:** why your circuits and mitigation work, what they cost, and what they cannot fix. Expect follow-up questions.
- **Learning:** what you tried that failed and what you concluded from it. A clear account of a negative result is valuable.
- **Presentation:** a clear story, with plots of your own data.

AI assistants are allowed as tools. The judges are assessing *your* understanding, so be ready to explain every step yourself.

## References

**The central question: trusting results you cannot check**

- P. Hauke *et al.*, [Can one trust quantum simulators?](https://doi.org/10.1088/0034-4885/75/8/082401), *Rep. Prog. Phys.* 75, 082401 (2012). The classic statement of the problem.
- Barron *et al.* (IBM), [Observable estimation in the absence of classical verification](https://arxiv.org/abs/2607.25998), arXiv:2607.25998 (2026). Consistency experiments, an operator Loschmidt echo, and noise-model validation in place of a classical answer. This is the closest paper to this challenge.
- J. Carrasco, A. Elben, C. Kokail, B. Kraus, P. Zoller, [Theoretical and experimental perspectives of quantum verification](https://arxiv.org/abs/2102.05927), *PRX Quantum* 2, 010102 (2021); J. Eisert *et al.*, [Quantum certification and benchmarking](https://doi.org/10.1038/s42254-020-0186-4), *Nat. Rev. Phys.* 2, 382 (2020).

**Claims and rebuttals**

- Y. Kim *et al.* (IBM), [Evidence for the utility of quantum computing before fault tolerance](https://doi.org/10.1038/s41586-023-06096-3), *Nature* 618, 500 (2023), and the classical responses: J. Tindall *et al.*, [Efficient tensor network simulation of IBM's Eagle kicked Ising experiment](https://arxiv.org/abs/2306.14887), *PRX Quantum* 5, 010308 (2024); T. Begušić, J. Gray, G. K.-L. Chan, [Fast and converged classical simulations of evidence for the utility of quantum computing before fault tolerance](https://arxiv.org/abs/2308.05077), *Sci. Adv.* 10, eadk4321 (2024); S. Anand *et al.*, [Classical benchmarking of zero noise extrapolation beyond the exactly-verifiable regime](https://arxiv.org/abs/2306.17839) (2023).
- A. D. King *et al.* (D-Wave), [Beyond-classical computation in quantum simulation](https://doi.org/10.1126/science.ado6285), *Science* 388, 199 (2025), and J. Tindall *et al.*, [Dynamics of disordered quantum systems with two- and three-dimensional tensor networks](https://doi.org/10.1126/science.adx2728), *Science* 392, 868 (2026).
- R. Haghshenas *et al.* (Quantinuum), [Digital quantum magnetism on a trapped-ion quantum computer](https://doi.org/10.1038/s41586-026-10445-3), *Nature* 653, 56 (2026): 2D transverse-field Ising dynamics on trapped ions, essentially stage 3 at larger scale. Classical counterpart: T. Begušić, G. K.-L. Chan, [Real-time operator evolution in two and three dimensions via sparse Pauli dynamics](https://arxiv.org/abs/2409.03097), *PRX Quantum* 6, 020302 (2025).
- Google Quantum AI, [Observation of constructive interference at the edge of quantum ergodicity](https://doi.org/10.1038/s41586-025-09526-6), *Nature* 646, 825 (2025): an echo-based observable presented as *verifiable* by repeating it on another device.

**Verification tools**

- A. Elben *et al.*, [Cross-platform verification of intermediate scale quantum devices](https://doi.org/10.1103/PhysRevLett.124.010504), *PRL* 124, 010504 (2020).
- R. Shaffer *et al.*, [Practical verification protocols for analog quantum simulators](https://doi.org/10.1038/s41534-021-00380-8), *npj Quantum Inf.* 7, 46 (2021): includes time-reversal (echo) checks on trapped ions.
- C. Kokail *et al.*, [Self-verifying variational quantum simulation of lattice models](https://doi.org/10.1038/s41586-019-1177-4), *Nature* 569, 355 (2019).
- L. Pastori *et al.*, [Characterization and verification of Trotterized digital quantum simulation via Hamiltonian and Liouvillian learning](https://doi.org/10.1103/PRXQuantum.3.030324), *PRX Quantum* 3, 030324 (2022).
- A. L. Shaw *et al.*, [Benchmarking highly entangled states on a 60-atom analogue quantum simulator](https://doi.org/10.1038/s41586-024-07173-x), *Nature* 628, 71 (2024): extrapolating fidelity from checkable regimes.
- T. Proctor *et al.*, [Measuring the capabilities of quantum computers](https://doi.org/10.1038/s41567-021-01409-7), *Nat. Phys.* 18, 75 (2022): mirror circuits.

**Error mitigation and Trotter error**

- Z. Cai *et al.*, [Quantum error mitigation](https://doi.org/10.1103/RevModPhys.95.045005), *Rev. Mod. Phys.* 95, 045005 (2023): the standard review.
- K. Temme, S. Bravyi, J. Gambetta, [Error mitigation for short-depth quantum circuits](https://doi.org/10.1103/PhysRevLett.119.180509), *PRL* 119, 180509 (2017) (zero-noise extrapolation).
- X. Bonet-Monroig *et al.*, [Low-cost error mitigation by symmetry verification](https://doi.org/10.1103/PhysRevA.98.062339), *PRA* 98, 062339 (2018).
- J. J. Wallman, J. Emerson, [Noise tailoring for scalable quantum computation via randomized compiling](https://doi.org/10.1103/PhysRevA.94.052325), *PRA* 94, 052325 (2016).
- A. M. Childs *et al.*, [Theory of Trotter error with commutator scaling](https://doi.org/10.1103/PhysRevX.11.011020), *PRX* 11, 011020 (2021).

**IonQ**

- [IonQ documentation](https://docs.ionq.com/), [job cost and usage](https://docs.ionq.com/guides/job-cost-and-usage), [native gates](https://docs.ionq.com/features/getting-started-with-native-gates), [noise-model simulation](https://docs.ionq.com/features/simulation-with-noise-models).
- Z. Davoudi, C.-C. Hsieh, S. V. Kadam, [Quantum computation of hadron scattering in a lattice gauge theory](https://arxiv.org/abs/2505.20408) (2025): digitized time evolution on IonQ Forte, with symmetry-based mitigation.